In [ ]:
RANDOM_STATE = 42
OUT_DIR = "runs2"
RUN_NAME = "elliptic"

In [ ]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from scipy.stats import randint, uniform




DATA_DIR = r"D:\elliptic\Elliptic_Dataset"
TXS_FEATURES_FILE = "txs_features.csv"
TXS_CLASSES_FILE  = "txs_classes.csv"
TXS_EDGELIST_FILE = "txs_edgelist.csv"  

In [ ]:

import os

df_txs_features = pd.read_csv(os.path.join(DATA_DIR, TXS_FEATURES_FILE))
df_txs_classes  = pd.read_csv(os.path.join(DATA_DIR, TXS_CLASSES_FILE))

print("txs_features shape:", df_txs_features.shape)
print("txs_classes  shape:", df_txs_classes.shape)
print("txs_features columns:", df_txs_features.columns.tolist())
print("txs_classes  columns:", df_txs_classes.columns.tolist())


txs_features shape: (203769, 184)
txs_classes  shape: (203769, 2)
txs_features columns: ['txId', 'Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10', 'Local_feature_11', 'Local_feature_12', 'Local_feature_13', 'Local_feature_14', 'Local_feature_15', 'Local_feature_16', 'Local_feature_17', 'Local_feature_18', 'Local_feature_19', 'Local_feature_20', 'Local_feature_21', 'Local_feature_22', 'Local_feature_23', 'Local_feature_24', 'Local_feature_25', 'Local_feature_26', 'Local_feature_27', 'Local_feature_28', 'Local_feature_29', 'Local_feature_30', 'Local_feature_31', 'Local_feature_32', 'Local_feature_33', 'Local_feature_34', 'Local_feature_35', 'Local_feature_36', 'Local_feature_37', 'Local_feature_38', 'Local_feature_39', 'Local_feature_40', 'Local_feature_41', 'Local_feature_42', 'Local_feature_43', 'Local_feature_44', 'Local_feature_45',

In [ ]:


df = df_txs_features.merge(df_txs_classes, on="txId", how="left")

if "class" not in df.columns:
    raise ValueError("Không tìm thấy cột 'class' sau khi merge!")

print("\nPhân bố class ban đầu:")
print(df["class"].value_counts(dropna=False))


df = df.dropna(subset=["class"]).copy()
df["class"] = df["class"].astype(int)
df = df[df["class"] != 3].copy()


df["label"] = (df["class"] == 2).astype(int)

LABEL_COL = "label"
print("\nPhân bố label sau khi bỏ unknown (0=licit,1=illicit):")
print(df[LABEL_COL].value_counts())


Phân bố class ban đầu:
class
3    157205
2     42019
1      4545
Name: count, dtype: int64

Phân bố label sau khi bỏ unknown (0=licit,1=illicit):
label
1    42019
0     4545
Name: count, dtype: int64


In [ ]:

possible_ts_cols = ["Time step", "time_step"]
ts_col = None
for c in possible_ts_cols:
    if c in df.columns:
        ts_col = c
        break

if ts_col is None:
    raise ValueError("Không tìm thấy cột time-step (ví dụ 'Time step' hoặc 'time_step')!")

drop_id_cols = ["txId", ts_col]
drop_label_cols = ["class", LABEL_COL]

cols_to_drop = [c for c in drop_id_cols + drop_label_cols if c in df.columns]

feature_cols = [c for c in df.columns if c not in cols_to_drop]

txid = df["txId"].astype(int).values

X_df = df[feature_cols].copy()
y = df[LABEL_COL].values
time_steps = df[ts_col].values

print("\nSố feature:", len(feature_cols))
print("Một vài feature đầu:", feature_cols[:10])


num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
X_df = X_df[num_cols].copy()
print("\nSố feature numeric:", len(num_cols))


Số feature: 182
Một vài feature đầu: ['Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10']

Số feature numeric: 182


In [ ]:

unique_ts = np.sort(df[ts_col].unique())
n_ts = len(unique_ts)
print("\nSố time-step khác nhau:", n_ts)
print("Các time-step đầu:", unique_ts[:10], "...", unique_ts[-10:])

if n_ts == 49:
    train_ts = unique_ts[:30]
    val_ts   = unique_ts[30:34]
    test_ts  = unique_ts[34:]
elif n_ts >= 50:
    train_ts = unique_ts[:30]
    val_ts   = unique_ts[30:40]
    test_ts  = unique_ts[40:50]
else:

    idx_train_end = int(0.6 * n_ts)
    idx_val_end   = int(0.8 * n_ts)
    train_ts = unique_ts[:idx_train_end]
    val_ts   = unique_ts[idx_train_end:idx_val_end]
    test_ts  = unique_ts[idx_val_end:]

print("\nTime-step TRAIN:", train_ts[0], "->", train_ts[-1])
print("Time-step VAL  :", val_ts[0],   "->", val_ts[-1])
print("Time-step TEST :", test_ts[0],  "->", test_ts[-1])

train_mask = df[ts_col].isin(train_ts)
val_mask   = df[ts_col].isin(val_ts)
test_mask  = df[ts_col].isin(test_ts)

X_train = X_df[train_mask].values
y_train = y[train_mask]

X_val   = X_df[val_mask].values
y_val   = y[val_mask]

X_test  = X_df[test_mask].values
y_test  = y[test_mask]


txid_train = txid[train_mask]
txid_val   = txid[val_mask]
txid_test  = txid[test_mask]
print("txid_test sample:", txid_test[:5])

print("\nKích thước:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

print("\nPhân bố nhãn train:")
print(pd.Series(y_train).value_counts())
print("\nPhân bố nhãn val:")
print(pd.Series(y_val).value_counts())
print("\nPhân bố nhãn test:")
print(pd.Series(y_test).value_counts())


Số time-step khác nhau: 49
Các time-step đầu: [ 1  2  3  4  5  6  7  8  9 10] ... [40 41 42 43 44 45 46 47 48 49]

Time-step TRAIN: 1 -> 30
Time-step VAL  : 31 -> 34
Time-step TEST : 35 -> 49
txid_test sample: [1813992 2234847 2364754 2365784 2503693]

Kích thước:
X_train: (26905, 182) y_train: (26905,)
X_val  : (2989, 182) y_val  : (2989,)
X_test : (16670, 182) y_test : (16670,)

Phân bố nhãn train:
1    23951
0     2954
Name: count, dtype: int64

Phân bố nhãn val:
1    2481
0     508
Name: count, dtype: int64

Phân bố nhãn test:
1    15587
0     1083
Name: count, dtype: int64


In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)


In [ ]:

import os
import numpy as np

def load_saved_proba(save_dirs, filename, txid_test, fill=0.5):
    """
    Load npz {txid, proba, (optional) threshold} và align theo txid_test.
    Trả về: (proba_aligned or None, threshold, used_file)
    """
    for d in save_dirs:
        f = os.path.join(d, filename)
        if os.path.exists(f):
            npz = np.load(f, allow_pickle=True)
            txid = npz["txid"].astype(int)
            proba = npz["proba"].astype(float)
            thr = float(npz["threshold"]) if "threshold" in npz.files else 0.5

            mp = {int(t): float(p) for t, p in zip(txid.tolist(), proba.tolist())}
            aligned = np.array([mp.get(int(t), np.nan) for t in txid_test], dtype=float)

            missing = int(np.isnan(aligned).sum())
            if missing > 0:
                print(f"[WARN] Missing {missing}/{len(aligned)} txId in {f}. Fill={fill}")
                aligned = np.nan_to_num(aligned, nan=fill)

            return aligned, thr, f

    return None, 0.5, None



GNN_SAVE_CANDIDATES = [
    "gnn_saved",
    r"D:\elliptic\ellipticv2\gnn_saved",
]
gnn_proba_test, gnn_threshold, gnn_file = load_saved_proba(
    GNN_SAVE_CANDIDATES, "gnn_test_preds.npz", txid_test, fill=0.5
)
if gnn_proba_test is not None:
    print("[OK] Loaded GNN preds:", gnn_proba_test.shape, "| thr:", gnn_threshold, "| file:", gnn_file)
else:
    print("[INFO] Không tìm thấy GNN preds trong:", GNN_SAVE_CANDIDATES)



SAGE_SAVE_CANDIDATES = [
    "sage_saved",
    r"D:\elliptic\ellipticv2\sage_saved",
]
sage_proba_test, sage_threshold, sage_file = load_saved_proba(
    SAGE_SAVE_CANDIDATES, "gnn_test_preds.npz", txid_test, fill=0.5
)
if sage_proba_test is not None:
    print("[OK] Loaded SAGE preds:", sage_proba_test.shape, "| thr:", sage_threshold, "| file:", sage_file)
else:
    print("[INFO] Không tìm thấy SAGE preds trong:", SAGE_SAVE_CANDIDATES)


[WARN] Missing 324/16670 txId in gnn_saved\gnn_test_preds.npz. Fill=0.5
[OK] Loaded GNN preds: (16670,) | thr: 0.5 | file: gnn_saved\gnn_test_preds.npz
[WARN] Missing 324/16670 txId in sage_saved\gnn_test_preds.npz. Fill=0.5
[OK] Loaded SAGE preds: (16670,) | thr: 0.5 | file: sage_saved\gnn_test_preds.npz


In [ ]:

from sklearn.metrics import roc_auc_score


def sample_param(dist, rng):
    """Lấy 1 giá trị từ distribution hoặc list."""
    if hasattr(dist, "rvs"):

        return dist.rvs(random_state=rng)

    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

def random_search_single_model(
    name,
    base_estimator,
    param_dist,
    X_train, y_train,
    X_val,   y_val,
    n_iter=30,
    scoring="macro"
):

    print(f"\n===== Random search cho {name} (không k-fold, dùng VAL) =====")
    rng = np.random.RandomState(RANDOM_STATE)
    best_f1 = -1.0
    best_params = None

    for i in range(n_iter):
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = clone(base_estimator)
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)

        y_val_score = None
        if hasattr(model, 'predict_proba'):
            try:
                y_val_score = model.predict_proba(X_val)[:, 1]
            except Exception:
                y_val_score = None
        elif hasattr(model, 'decision_function'):
            try:
                y_val_score = model.decision_function(X_val)
            except Exception:
                y_val_score = None

        auc_val = np.nan
        if y_val_score is not None:
            try:
                auc_val = roc_auc_score(y_val, y_val_score)
            except Exception:
                auc_val = np.nan

        f1_macro = f1_score(y_val, y_val_pred, average="macro")

        print(f"Iter {i+1:02d}/{n_iter}: F1_macro(val) = {f1_macro:.4f}, AUC(val) = {auc_val:.4f}, params = {params}")

        if f1_macro > best_f1:
            best_f1 = f1_macro
            best_params = params

    print(f"\n>>> {name} – best F1_macro(val) = {best_f1:.4f}")
    print("Best params:", best_params)


    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_model = clone(base_estimator)
    best_model.set_params(**best_params)
    best_model.fit(X_train_full, y_train_full)

    return best_model


n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
print("\nscale_pos_weight (train):", scale_pos_weight)


scale_pos_weight (train): 0.12333514258277316


In [ ]:


# 1) RandomForest
rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_param_dist = {
    "n_estimators": randint(200, 600),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 50),
    "min_samples_leaf": randint(1, 20),
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"],
    "criterion": ["gini", "entropy"],
}

best_rf = random_search_single_model(
    "RandomForest",
    rf_base,
    rf_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)







===== Random search cho RandomForest (không k-fold, dùng VAL) =====
Iter 01/30: F1_macro(val) = 0.9664, AUC(val) = 0.9957, params = {'n_estimators': 302, 'max_depth': 22, 'min_samples_split': 30, 'min_samples_leaf': 15, 'max_features': None, 'class_weight': 'balanced', 'criterion': 'gini'}
Iter 02/30: F1_macro(val) = 0.9486, AUC(val) = 0.9934, params = {'n_estimators': 220, 'max_depth': 9, 'min_samples_split': 20, 'min_samples_leaf': 11, 'max_features': None, 'class_weight': 'balanced', 'criterion': 'gini'}
Iter 03/30: F1_macro(val) = 0.9764, AUC(val) = 0.9968, params = {'n_estimators': 299, 'max_depth': 10, 'min_samples_split': 25, 'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': None, 'criterion': 'entropy'}
Iter 04/30: F1_macro(val) = 0.9704, AUC(val) = 0.9970, params = {'n_estimators': 543, 'max_depth': 14, 'min_samples_split': 31, 'min_samples_leaf': 6, 'max_features': 'log2', 'class_weight': 'balanced', 'criterion': 'entropy'}
Iter 05/30: F1_macro(val) = 0.8332, AU

In [ ]:
# 2) XGBoost
xgb_base = XGBClassifier(
    random_state=RANDOM_STATE,
    tree_method="hist",     
    eval_metric="logloss",
    use_label_encoder=False
)

xgb_param_dist = {
    "n_estimators": randint(200, 800),
    "max_depth": randint(3, 12),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),     
    "colsample_bytree": uniform(0.6, 0.4), 
    "min_child_weight": randint(1, 10),
    "reg_lambda": uniform(0, 5),
    "objective": ["binary:logistic"],     
    "eval_metric": ["logloss", "aucpr"],    
    "scale_pos_weight": [scale_pos_weight]
}

best_xgb = random_search_single_model(
    "XGBoost",
    xgb_base,
    xgb_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho XGBoost (không k-fold, dùng VAL) =====


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:23:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 01/30: F1_macro(val) = 0.9682, AUC(val) = 0.9967, params = {'n_estimators': 302, 'max_depth': 6, 'learning_rate': np.float64(0.28570714885887566), 'subsample': np.float64(0.892797576724562), 'colsample_bytree': np.float64(0.8394633936788146), 'min_child_weight': 7, 'reg_lambda': np.float64(2.229163764267956), 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'scale_pos_weight': np.float64(0.12333514258277316)}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:23:37] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 02/30: F1_macro(val) = 0.9435, AUC(val) = 0.9939, params = {'n_estimators': 530, 'max_depth': 10, 'learning_rate': np.float64(0.10677549723031632), 'subsample': np.float64(0.6571467271687763), 'colsample_bytree': np.float64(0.8603553891795411), 'min_child_weight': 5, 'reg_lambda': np.float64(4.8495492608099715), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.12333514258277316)}


...

d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:25:16] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 29/30: F1_macro(val) = 0.9823, AUC(val) = 0.9981, params = {'n_estimators': 330, 'max_depth': 6, 'learning_rate': np.float64(0.267255063036884), 'subsample': np.float64(0.9119502183430496), 'colsample_bytree': np.float64(0.8568126584617151), 'min_child_weight': 1, 'reg_lambda': np.float64(0.8081435704730688), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.12333514258277316)}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:25:18] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 30/30: F1_macro(val) = 0.9437, AUC(val) = 0.9938, params = {'n_estimators': 303, 'max_depth': 6, 'learning_rate': np.float64(0.17839912021657184), 'subsample': np.float64(0.7489131066246972), 'colsample_bytree': np.float64(0.9760533769831113), 'min_child_weight': 9, 'reg_lambda': np.float64(0.025307919231093434), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.12333514258277316)}

>>> XGBoost – best F1_macro(val) = 0.9823
Best params: {'n_estimators': 330, 'max_depth': 6, 'learning_rate': np.float64(0.267255063036884), 'subsample': np.float64(0.9119502183430496), 'colsample_bytree': np.float64(0.8568126584617151), 'min_child_weight': 1, 'reg_lambda': np.float64(0.8081435704730688), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.12333514258277316)}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:25:19] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [12]:
# 3) LightGBM
lgbm_base = LGBMClassifier(
    random_state=RANDOM_STATE,
    objective="binary",
    n_jobs=-1
)

lgbm_param_dist = {
    "n_estimators": randint(200, 800),
    "num_leaves": randint(15, 255),
    "max_depth": randint(3, 12),            # -1 = no limit
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),          # bagging_fraction
    "colsample_bytree": uniform(0.6, 0.4),   # feature_fraction
    "min_child_samples": randint(10, 100),
    "reg_lambda": uniform(0, 5),
    # --------- "criterion" kiểu LightGBM: objective + metric ----------
    "objective": ["binary"],                      # có thể thêm "xentropy"
    "metric": ["binary_logloss", "auc", "aucpr"],  # AutoML tự chọn
    "scale_pos_weight": [scale_pos_weight]
} 

best_lgbm = random_search_single_model(
    "LightGBM",
    lgbm_base,
    lgbm_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho LightGBM (không k-fold, dùng VAL) =====
[LightGBM] [Info] Number of positive: 23951, number of negative: 2954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011398 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41708
[LightGBM] [Info] Number of data points in the train set: 26905, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.890206 -> initscore=2.092850
[LightGBM] [Info] Start training from score 2.092850
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 23951, number of negative: 2954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41708
[LightGBM] [Info] Number of data points in the train set: 26905, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.890206 -> initscore=2.092850
[LightGBM] [Info] Start training from score 2.092850
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 23951, number of negative: 2954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015435 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41708
[LightGBM] [Info] Number of data points in the train set: 26905, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.890206 -> initscore=2.092850
[LightGBM] [Info] Start training from score 2.092850
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

...

Iter 28/30: F1_macro(val) = 0.9802, AUC(val) = 0.9983, params = {'n_estimators': 703, 'num_leaves': 142, 'max_depth': 10, 'learning_rate': np.float64(0.2562747890433116), 'subsample': np.float64(0.8630451569201374), 'colsample_bytree': np.float64(0.8273234413341887), 'min_child_samples': 85, 'reg_lambda': np.float64(2.5440703841938), 'objective': 'binary', 'metric': 'auc', 'scale_pos_weight': np.float64(0.12333514258277316)}
[LightGBM] [Info] Number of positive: 23951, number of negative: 2954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010795 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41708
[LightGBM] [Info] Number of data points in the train set: 26905, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.890206 -> initscore=2.092850
[LightGBM] [Info] Start training from score 2.092850
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Iter 29/30: F1_macro(val) = 0.9818, AUC(val) = 0.9986, params = {'n_estimators': 574, 'num_leaves': 36, 'max_depth': 8, 'learning_rate': np.float64(0.2938789288997526), 'subsample': np.float64(0.794696861183782), 'colsample_bytree': np.float64(0.9624395150874216), 'min_child_samples': 36, 'reg_lambda': np.float64(3.974056517708242), 'objective': 'binary', 'metric': 'auc', 'scale_pos_weight': np.float64(0.12333514258277316)}
[LightGBM] [Info] Number of positive: 23951, number of negative: 2954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011607 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41708
[LightGBM] [Info] Number of data points in the train set: 26905, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.890206 -> initscore=2.092850
[LightGBM] [Info] Start training from score 2.092850
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Lig

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 26432, number of negative: 3462
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011620 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41915
[LightGBM] [Info] Number of data points in the train set: 29894, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.884191 -> initscore=2.032729
[LightGBM] [Info] Start training from score 2.032729
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [ ]:

from itertools import combinations
from sklearn.metrics import roc_auc_score

base_models = {
    "RF": best_rf,
    "XGB": best_xgb,
    "LGBM": best_lgbm
}


class StaticProbaModel:
    """Model giả lập sklearn cho ensemble (có predict_proba + predict)."""
    def __init__(self, proba_pos, threshold=0.5):
        self.proba_pos = np.asarray(proba_pos, dtype=float).ravel()
        self.threshold = float(threshold)

    def predict_proba(self, X):
        n = len(X)
        if n != len(self.proba_pos):
            raise ValueError(f"Length mismatch: X has {n} rows but proba_pos has {len(self.proba_pos)}")
        p = self.proba_pos
        return np.vstack([1 - p, p]).T

    def predict(self, X):

        n = len(X)
        if n != len(self.proba_pos):
            raise ValueError(f"Length mismatch: X has {n} rows but proba_pos has {len(self.proba_pos)}")
        return (self.proba_pos >= self.threshold).astype(int)


if gnn_proba_test is not None:
    base_models["GNN"] = StaticProbaModel(gnn_proba_test)
    print("[OK] Added GNN into base_models for ensemble.")
if sage_proba_test is not None:
    base_models["SAGE"] = StaticProbaModel(sage_proba_test)
    print("[OK] Added SAGE into base_models for ensemble.")


def _proba_pos(model, X):
    proba = model.predict_proba(X)
    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]
    return proba.ravel()

def ensemble_predict(models_dict, X, voting="soft", threshold=0.5):
    models = list(models_dict.values())

    probas = np.vstack([_proba_pos(m, X) for m in models])  
    proba_mean = probas.mean(axis=0)

    if voting == "soft":
        y_pred = (proba_mean >= threshold).astype(int)
        return y_pred, proba_mean

    if voting == "hard":
        preds = np.vstack([m.predict(X) for m in models]).astype(int)
        votes = preds.sum(axis=0)
        half = len(models) / 2

        y_pred = (votes > half).astype(int)


        tie_mask = (votes == half)
        if np.any(tie_mask):
            y_pred[tie_mask] = (proba_mean[tie_mask] >= threshold).astype(int)

        return y_pred, proba_mean

    raise ValueError("voting phải là 'hard' hoặc 'soft'")

def eval_binary(y_true, y_pred, y_score=None):
    out = {}
    out["accuracy"] = accuracy_score(y_true, y_pred)
    out["f1_binary"] = f1_score(y_true, y_pred)
    out["f1_micro"]  = f1_score(y_true, y_pred, average="micro")
    out["f1_macro"]  = f1_score(y_true, y_pred, average="macro")
    if y_score is not None:
        try:
            out["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            out["roc_auc"] = np.nan
    return out

results = []


from itertools import combinations

model_order = [k for k in ["RF", "XGB", "LGBM", "GNN","SAGE"] if k in base_models]

combos = []
for r in range(2, len(model_order) + 1): 
    combos.extend(list(combinations(model_order, r)))

print("Total combos =", len(combos))  

for combo in combos:
    combo_models = {k: base_models[k] for k in combo}
    combo_name = "+".join(combo)

    for voting in ["hard", "soft"]:
        y_pred, y_score = ensemble_predict(combo_models, X_test_scaled, voting=voting, threshold=0.5)
        m = eval_binary(y_test, y_pred, y_score)
        
        results.append({"ensemble": combo_name, "voting": voting, **m})

        print(f"\n===== Ensemble [{combo_name}] | {voting.upper()} vote =====")
        print(classification_report(y_test, y_pred, digits=6))
        print(f"Accuracy: {m['accuracy']:.6f}")
        print(f"F1 (binary, pos_label=1): {m['f1_binary']:.6f}")
        print(f"F1 micro: {m['f1_micro']:.6f}")
        print(f"F1 macro: {m['f1_macro']:.6f}")
        ra = m.get("roc_auc", np.nan)
        print(f"ROC-AUC: {ra:.6f}" if np.isfinite(ra) else "ROC-AUC: nan")
        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values(
    by=["f1_binary", "roc_auc", "accuracy"],
    ascending=False
)
display(results_df)


[OK] Added GNN into base_models for ensemble.
[OK] Added SAGE into base_models for ensemble.
Total combos = 26

===== Ensemble [RF+XGB] | HARD vote =====
              precision    recall  f1-score   support

           0   0.884058  0.732225  0.801010      1083
           1   0.981614  0.993328  0.987436     15587

    accuracy                       0.976365     16670
   macro avg   0.932836  0.862777  0.894223     16670
weighted avg   0.975276  0.976365  0.975325     16670

Accuracy: 0.976365
F1 (binary, pos_label=1): 0.987436
F1 micro: 0.976365
F1 macro: 0.894223
ROC-AUC: 0.923969
Confusion matrix:
[[  793   290]
 [  104 15483]]

===== Ensemble [RF+XGB] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.884058  0.732225  0.801010      1083
           1   0.981614  0.993328  0.987436     15587

    accuracy                       0.976365     16670
   macro avg   0.932836  0.862777  0.894223     16670
weighted avg   0.975276  0.976365  0.975325  

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+LGBM] | HARD vote =====
              precision    recall  f1-score   support

           0   0.923256  0.733149  0.817293      1083
           1   0.981720  0.995766  0.988693     15587

    accuracy                       0.978704     16670
   macro avg   0.952488  0.864457  0.902993     16670
weighted avg   0.977922  0.978704  0.977558     16670

Accuracy: 0.978704
F1 (binary, pos_label=1): 0.988693
F1 micro: 0.978704
F1 macro: 0.902993
ROC-AUC: 0.921331
Confusion matrix:
[[  794   289]
 [   66 15521]]

===== Ensemble [RF+LGBM] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.923256  0.733149  0.817293      1083
           1   0.981720  0.995766  0.988693     15587

    accuracy                       0.978704     16670
   macro avg   0.952488  0.864457  0.902993     16670
weighted avg   0.977922  0.978704  0.977558     16670

Accuracy: 0.978704
F1 (binary, pos_label=1): 0.988693
F1 micro: 0.978704
F1 macro: 0.902993
ROC-AUC

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+GNN] | HARD vote =====
              precision    recall  f1-score   support

           0   0.988796  0.651893  0.785754      1083
           1   0.976373  0.999487  0.987794     15587

    accuracy                       0.976905     16670
   macro avg   0.982584  0.825690  0.886774     16670
weighted avg   0.977180  0.976905  0.974668     16670

Accuracy: 0.976905
F1 (binary, pos_label=1): 0.987794
F1 micro: 0.976905
F1 macro: 0.886774
ROC-AUC: 0.919535
Confusion matrix:
[[  706   377]
 [    8 15579]]

===== Ensemble [RF+GNN] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.988796  0.651893  0.785754      1083
           1   0.976373  0.999487  0.987794     15587

    accuracy                       0.976905     16670
   macro avg   0.982584  0.825690  0.886774     16670
weighted avg   0.977180  0.976905  0.974668     16670

Accuracy: 0.976905
F1 (binary, pos_label=1): 0.987794
F1 micro: 0.976905
F1 macro: 0.886774
ROC-AUC: 

...

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [XGB+LGBM+GNN+SAGE] | HARD vote =====
              precision    recall  f1-score   support

           0   0.969735  0.710065  0.819829      1083
           1   0.980223  0.998460  0.989258     15587

    accuracy                       0.979724     16670
   macro avg   0.974979  0.854262  0.904543     16670
weighted avg   0.979542  0.979724  0.978250     16670

Accuracy: 0.979724
F1 (binary, pos_label=1): 0.989258
F1 micro: 0.979724
F1 macro: 0.904543
ROC-AUC: 0.919246
Confusion matrix:
[[  769   314]
 [   24 15563]]

===== Ensemble [XGB+LGBM+GNN+SAGE] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.969735  0.710065  0.819829      1083
           1   0.980223  0.998460  0.989258     15587

    accuracy                       0.979724     16670
   macro avg   0.974979  0.854262  0.904543     16670
weighted avg   0.979542  0.979724  0.978250     16670

Accuracy: 0.979724
F1 (binary, pos_label=1): 0.989258
F1 micro: 0.979724
F1 mac

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+XGB+LGBM+GNN+SAGE] | HARD vote =====
              precision    recall  f1-score   support

           0   0.974042  0.727608  0.832981      1083
           1   0.981401  0.998653  0.989952     15587

    accuracy                       0.981044     16670
   macro avg   0.977721  0.863131  0.911466     16670
weighted avg   0.980923  0.981044  0.979754     16670

Accuracy: 0.981044
F1 (binary, pos_label=1): 0.989952
F1 micro: 0.981044
F1 macro: 0.911466
ROC-AUC: 0.916519
Confusion matrix:
[[  788   295]
 [   21 15566]]

===== Ensemble [RF+XGB+LGBM+GNN+SAGE] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.978908  0.728532  0.835363      1083
           1   0.981467  0.998909  0.990112     15587

    accuracy                       0.981344     16670
   macro avg   0.980188  0.863721  0.912737     16670
weighted avg   0.981301  0.981344  0.980058     16670

Accuracy: 0.981344
F1 (binary, pos_label=1): 0.990112
F1 micro: 0.981344


,ensemble,voting,accuracy,f1_binary,f1_micro,f1_macro,roc_auc
51,RF+XGB+LGBM+GNN+SAGE,soft,0.981344,0.990112,0.981344,0.912737,0.916519
27,RF+LGBM+GNN,soft,0.981284,0.990080,0.981284,0.912413,0.920650
29,RF+LGBM+SAGE,soft,0.981224,0.990049,0.981224,0.912088,0.907528
25,RF+XGB+SAGE,soft,0.981224,0.990048,0.981224,0.912176,0.910037
23,RF+XGB+GNN,soft,0.981164,0.990015,0.981164,0.912027,0.921534
50,RF+XGB+LGBM+GNN+SAGE,hard,0.981044,0.989952,0.981044,0.911466,0.916519
26,RF+LGBM+GNN,hard,0.980984,0.989922,0.980984,0.910876,0.920650
28,RF+LGBM+SAGE,hard,0.980984,0.989922,0.980984,0.910876,0.907528
22,RF+XGB+GNN,hard,0.980864,0.989857,0.980864,0.910581,0.921534
24,RF+XGB+SAGE,hard,0.980744,0.989792,0.980744,0.910110,0.910037


In [ ]:

def evaluate_on_test(name, model, X_test, y_test):
    print(f"\n===== {name} trên TEST =====")
    y_pred = model.predict(X_test)

    y_score = None
    if hasattr(model, 'predict_proba'):
        try:
            y_score = model.predict_proba(X_test)[:, 1]
        except Exception:
            y_score = None
    elif hasattr(model, 'decision_function'):
        try:
            y_score = model.decision_function(X_test)
        except Exception:
            y_score = None

    auc = np.nan
    if y_score is not None:
        try:
            auc = roc_auc_score(y_test, y_score)
        except Exception:
            auc = np.nan
    acc      = accuracy_score(y_test, y_pred)
    f1_bin   = f1_score(y_test, y_pred)            
    f1_micro = f1_score(y_test, y_pred, average="micro")
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy : {acc:.6f}")
    print(f"F1 (binary, pos_label=1): {f1_bin:.6f}")
    print(f"F1 micro : {f1_micro:.6f}")
    print(f"F1 macro : {f1_macro:.6f}")
    print(f"ROC-AUC  : {auc:.6f}")

    print("\nclassification_report:")
    print(classification_report(y_test, y_pred, digits=6))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


models = {
    "RandomForest": best_rf,
    "XGBoost":      best_xgb,
    "LightGBM":     best_lgbm,
}

for name, model in models.items():
    evaluate_on_test(name, model, X_test_scaled, y_test)

if "sage_proba_test" in globals() and sage_proba_test is not None:
    thr = sage_threshold if (("sage_threshold" in globals()) and (sage_threshold is not None)) else 0.5
    sage_pred = (sage_proba_test >= thr).astype(int)
    try:
        sage_auc = roc_auc_score(y_test, sage_proba_test)
    except Exception:
        sage_auc = np.nan
    print("\n===== GraphSAGE (loaded proba) trên TEST =====")
    print(f"Accuracy : {accuracy_score(y_test, sage_pred):.6f}")
    print(f"F1 macro : {f1_score(y_test, sage_pred, average='macro'):.6f}")
    print(f"ROC-AUC  : {sage_auc:.6f}")



===== RandomForest trên TEST =====
Accuracy : 0.979904
F1 (binary, pos_label=1): 0.989349
F1 micro : 0.979904
F1 macro : 0.905909
ROC-AUC  : 0.919847

classification_report:
              precision    recall  f1-score   support

           0   0.965174  0.716528  0.822470      1083
           1   0.980650  0.998204  0.989349     15587

    accuracy                       0.979904     16670
   macro avg   0.972912  0.857366  0.905909     16670
weighted avg   0.979645  0.979904  0.978508     16670

Confusion matrix:
[[  776   307]
 [   28 15559]]

===== XGBoost trên TEST =====
Accuracy : 0.972106
F1 (binary, pos_label=1): 0.985133
F1 micro : 0.972106
F1 macro : 0.879866
ROC-AUC  : 0.934410

classification_report:
              precision    recall  f1-score   support

           0   0.815306  0.737765  0.774600      1083
           1   0.981899  0.988388  0.985133     15587

    accuracy                       0.972106     16670
   macro avg   0.898603  0.863077  0.879866     16670
weighte

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:

try:
    import shap
    shap_available = True
except Exception as e:
    print("Không thể import SHAP (shap). Hãy cài: pip install shap")
    print(e)
    shap_available = False

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if shap_available and X_test_scaled.shape[0] > 0:
    shap.initjs()


    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Liberation Serif']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['xtick.labelsize'] = 11
    plt.rcParams['ytick.labelsize'] = 11
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['figure.titlesize'] = 18


    n_background = min(200,X_train_scaled.shape[0])
    n_explain    = min(200,X_test_scaled.shape[0])

    X_background = X_train_scaled[:n_background]
    X_explain    = X_test_scaled[:n_explain]

    print(f"[Elliptic] Background SHAP size: {X_background.shape}")
    print(f"[Elliptic] Explain SHAP size   : {X_explain.shape}")

    models_to_explain = {
        "RandomForest": best_rf,
        "XGBoost":      best_xgb,
        "LightGBM":     best_lgbm,
    }

    mean_abs_dict = {}   

    for name, model in models_to_explain.items():
        print(f"\n================ SHAP cho {name} (Elliptic++) ================")


        explainer = shap.TreeExplainer(
            model,
            data=X_background,
            feature_perturbation="interventional",
            model_output="probability"
        )

        sv = explainer.shap_values(
            X_explain,
            check_additivity=False
        )

        if isinstance(sv, list):
            if hasattr(model, "classes_"):
                cls_idx = int(np.where(model.classes_ == 1)[0][0])
            else:
                cls_idx = 1
            sv_pos = np.array(sv[cls_idx])

        elif isinstance(sv, np.ndarray) and sv.ndim == 3:
            if hasattr(model, "classes_"):
                cls_idx = int(np.where(model.classes_ == 1)[0][0])
            else:
                cls_idx = sv.shape[2] - 1
            sv_pos = np.array(sv[:, :, cls_idx])

        else:
            sv_pos = np.array(sv)

        assert sv_pos.ndim == 2, f"sv_pos không phải 2D cho model {name} (shape={sv_pos.shape})"
        print("  sv_pos.shape =", sv_pos.shape, "| ndim =", sv_pos.ndim)


        mean_abs = np.mean(np.abs(sv_pos), axis=0)  
        mean_abs_dict[name] = mean_abs


    df_mean = pd.DataFrame(mean_abs_dict, index=feature_cols)


    top_n = 10 if len(feature_cols) >= 10 else len(feature_cols)
    df_mean['avg'] = df_mean.mean(axis=1)
    df_mean = df_mean.sort_values('avg', ascending=False)
    df_top = df_mean.head(top_n).drop(columns='avg')   


    plt.close('all')
    fig, axes = plt.subplots(
        1, 2,
        figsize=(11, 4.3),   
        dpi=600,
        constrained_layout=True
    )
    ax0, ax1 = axes

  
    sns.heatmap(
        df_top,
        annot=True,
        fmt=".4f",              
        cmap="YlOrRd",
        linewidths=0.5,
        linecolor='gray',
        cbar_kws={"label": "Mean", "shrink": 0.8},
        ax=ax0,
        annot_kws={"size": 11}
    )

    ax0.set_title("(a) Elliptic++ Heatmap", pad=10, fontweight='bold')
    ax0.set_xlabel("")
    ax0.set_ylabel("")
    ax0.set_yticklabels(df_top.index, rotation=0)
    ax0.set_xticklabels(df_top.columns, rotation=0)


    df_long = (
        df_top
        .reset_index()
        .melt(id_vars="index", var_name="Model", value_name="Importance")
        .rename(columns={"index": "Feature"})
    )

    feature_order = df_top.index.tolist()

    sns.barplot(
        data=df_long,
        x="Importance",
        y="Feature",
        hue="Model",
        order=feature_order,
        palette=["#1f77b4", "#d62728", "#2ca02c"],  
        ax=ax1,
        edgecolor='black',
        linewidth=0.8
    )

    ax1.set_title("(b) Elliptic++", pad=10, fontweight='bold')
    ax1.set_xlabel("Mean")
    ax1.set_ylabel("")
    ax1.grid(axis="x", linestyle='--', alpha=0.5)

    ax1.legend(
        title="Model",
        frameon=True,
        fancybox=False,
        edgecolor='black'
    )

    plt.show()

else:
    print("Bỏ qua SHAP Elliptic++ vì không sẵn sàng hoặc không có mẫu test.")


In [ ]:

    df_mean = pd.DataFrame(mean_abs_dict, index=feature_cols)


    top_n = 6 if len(feature_cols) >= 10 else len(feature_cols)
    df_mean['avg'] = df_mean.mean(axis=1)
    df_mean = df_mean.sort_values('avg', ascending=False)
    df_top = df_mean.head(top_n).drop(columns='avg') 


    plt.close('all')
    fig, axes = plt.subplots(
        1, 2,
        figsize=(11, 4.3),  
        dpi=600,
        constrained_layout=True
    )
    ax0, ax1 = axes


    sns.heatmap(
        df_top,
        annot=True,
        fmt=".4f",          
        cmap="YlOrRd",
        linewidths=0.5,
        linecolor='gray',
        cbar_kws={"label": "Mean", "shrink": 0.8},
        ax=ax0,
        annot_kws={"size": 11}
    )

    ax0.set_title("(a) Elliptic++", pad=10, fontweight='bold')
    ax0.set_xlabel("")
    ax0.set_ylabel("")
    ax0.set_yticklabels(df_top.index, rotation=0)
    ax0.set_xticklabels(df_top.columns, rotation=0)


    df_long = (
        df_top
        .reset_index()
        .melt(id_vars="index", var_name="Model", value_name="Importance")
        .rename(columns={"index": "Feature"})
    )

    feature_order = df_top.index.tolist()

    sns.barplot(
        data=df_long,
        x="Importance",
        y="Feature",
        hue="Model",
        order=feature_order,
        palette=["#1f77b4", "#d62728", "#2ca02c"],  # giống code mẫu
        ax=ax1,
        edgecolor='black',
        linewidth=0.8
    )

    ax1.set_title("(b) Elliptic++", pad=10, fontweight='bold')
    ax1.set_xlabel("Mean")
    ax1.set_ylabel("")
    ax1.grid(axis="x", linestyle='--', alpha=0.5)

    ax1.legend(
        title="Model",
        frameon=True,
        fancybox=False,
        edgecolor='black'
    )

    fig.savefig("eliptic.png", dpi=600, bbox_inches="tight")




In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score

rows = []


base_models = {"RF": best_rf, "XGB": best_xgb, "LGBM": best_lgbm}
for name, model in base_models.items():
    y_pred = model.predict(X_test_scaled)
    y_score = None
    if hasattr(model, 'predict_proba'):
        try:
            y_score = model.predict_proba(X_test_scaled)[:, 1]
        except Exception:
            y_score = None
    elif hasattr(model, 'decision_function'):
        try:
            y_score = model.decision_function(X_test_scaled)
        except Exception:
            y_score = None

    auc = np.nan
    if y_score is not None:
        try:
            auc = roc_auc_score(y_test, y_score)
        except Exception:
            auc = np.nan
    rows.append({
        "seed": int(RANDOM_STATE),
        "model": name,
        "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
        "roc_auc": float(auc)
    })


if "results_df" in globals():
    tmp = results_df.copy()
    tmp["model"] = tmp["ensemble"].astype(str) + "_" + tmp["voting"].astype(str)
    for _, r in tmp.iterrows():
        rows.append({
            "seed": int(RANDOM_STATE),
            "model": str(r["model"]),
            "f1_macro": float(r["f1_macro"]),
            "roc_auc": float(r["roc_auc"]) if "roc_auc" in r else float(np.nan)
        })
else:
    print("WARNING: results_df chưa tồn tại -> chỉ lưu macro-F1 cho 3 base models.")


out_csv = Path(r"D:\elliptic\ellipticv2\runs2\macro_f1_all_models_by_seed.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

df_out = pd.DataFrame(rows)

write_header = (not out_csv.exists()) or out_csv.stat().st_size == 0
df_out.to_csv(out_csv, mode="a", header=write_header, index=False)

print("Saved:", out_csv, "| rows:", len(df_out))


Saved: D:\elliptic\ellipticv2\runs2\macro_f1_all_models_by_seed.csv | rows: 55


d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import os

out_dir = "outputs2"
os.makedirs(out_dir, exist_ok=True)

csv_path = os.path.join(out_dir, "ensemble_results.csv")
results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Saved:", csv_path)

Saved: outputs2\ensemble_results.csv
